In [1]:
import numpy as np
import pandas as pd

with open('regions') as f:
    regions = f.readlines()

In [85]:
def split_name(s, idk=1):
    name = ''
    idx = 0
    for i in s:
        if i[0].isdigit() or not i[0].isupper():
            break
        name += i + ' '
        idx += 1
        if idx >= 3:
            break
    if name and name[-2] == ',':
        name += s[idx]
    return [name[:-1]] + s[idx+idk:]

def split_name_n(s):
    name = ''
    for i in s[:3]:
        name += i + ' '
    return [name[:-1]] + s[3:]

def ensure_cols(n_cols, a):
    while len(a) > n_cols:
        a[-2] += ' ' + a[-1]
        a.pop()
    return a

def idk(a): # delete all ints after first from end of list
    out = []
    first = True
    for i in a[::-1]:
        if i and i[0].isdigit() and i[-1].isdigit() or i == '.':
            if not first:
                continue
            first = False
        out.append(i)
    return out[::-1]

def idk12(a):
    return [a[0]] + idk2(a[1:], name=False)

def idk2(a, name=True): # reg, name, ...
    reg = a[0]
    idx = 1
    if not a[1][0].isupper() or reg == 'Республика' or reg == 'Удмуртская':
        reg += ' ' + a[1]
        idx = 2
    if not a[2][0].isalpha() or reg == 'Республика Марий' or not a[2][0].isupper() or reg == 'Республика Северная':
        reg += ' ' + a[2]
        idx = 3
        if len(a[3]) == 1 and not a[3].isdigit():
            reg += ' ' + a[3] + ' ' + a[4]
            idx = 5
        elif len(a[2]) == 1:
            reg += ' ' + a[3]
            idx = 4
    if name:
        return [reg] + split_name(a[idx:])
    else:
        return [reg] + a[idx:]

def idk3(a):
    return [a[0], a[1], a[2], a[3], a[-2]]

def idk4(a):
    return [i for i in a.split() if i != '']

def parse_result(s):
    if s is None:
        return 0
    if 'III' in s:
        return 1
    if 'II' in s:
        return 2
    if 'I' in s or 'Победитель' in s:
        return 3
    return 0

def compr(a, n=1):
    s = ''
    for i in a[n:-1]:
        s += i + ' '
    return a[:n] + [s[:-1], a[-1]]

def trim_cols(n_cols, a):
    if a[-1] == 'Invited' or a[-1] == 'Reserve' or a[-1] == '1' and int(a[-2]) > 1:
        a = a[:-1]
    return a

def opt(a, n):
    if len(a) <= n:
        return np.nan
    if not a[n]:
        return np.nan
    return a[n]

regs = list(map(lambda s: compr(s.split()[1:] + [''], n=0)[0], regions))

def get_region(s):
    for i in regs:
        if i in s:
            return i
    return ''

def remove_times(a):
    return [i for i in a if ':' not in i]

def idk6(a):
    if a[2][0].isalpha():
        return [a[0], a[1] + ' ' + a[2]] + a[3:]

In [145]:
with open('belchonok') as f:
    bel = f.readlines()
df_bel = pd.DataFrame(map(lambda s: ensure_cols(3, idk(split_name(idk4(s)))), bel[4:]), columns=['name', 'total', 'result'])
df_bel = df_bel[(df_bel['name'] != '') & (df_bel['name'] != 'Работа')]
df_bel['result'] = df_bel['result'].map(parse_result)
df_bel.to_csv('belchonok.csv', index=False)
df_bel

,name,total,result
0,Хакимзянов Ильдар Марселевич,51,3
1,Хмелевский Максим Михайлович,48,3
2,Еникеев Кирилл Денисович,45,3
3,Главатских Виолика Игоревна,41,3
4,Савицкая Таисия Владимировна,40,3
...,...,...,...
1676,Бухштаб Максим Дмитриевич,50,1
1677,Богославец Таисия Андреевна,50,1
1678,Тошматов Азизджон Джамшедович,50,1
1679,Коробицына Дарья Викторовна,50,1


In [87]:
with open('granit_nauki') as f:
    gra = f.readlines()
df_gra = pd.DataFrame(map(lambda s: idk3(idk2(idk4(s)[1:])), gra), columns=['reg', 'name', 'grade', 'total', 'result'])
df_gra['result'] = df_gra['result'].map(parse_result)
df_gra.to_csv('granit_nauki.csv', index=False)
df_gra

,reg,name,grade,total,result
0,Самарская область,Федотов М. С.,10,"91,5",3
1,Московская область,Брылин М. И.,10,87,3
2,Воронежская область,Бубнов К. С.,10,78,2
3,Санкт-Петербург,Обиджанова А. С.,10,76,2
4,Новосибирская область,Лысаков Р. К.,10,76,2
...,...,...,...,...,...
721,Республика Башкортостан,Бердигулов Д. Ф.,11,1,0
722,Иностранный гражданин,Расулов А. Г.,11,1,0
723,Кемеровская область — Кузбасс,Шелепов М. Д.,11,1,0
724,Республика Татарстан (Татарстан),Ефремов О. А.,11,1,0


In [106]:
with open('innopolis_1') as f:
    inn1 = f.readlines()
df_inn1 = pd.DataFrame(map(lambda s: compr(idk(split_name(trim_cols(11, idk4(s)[1:])))), inn1), columns=['name', 'reg', 'total'])
df_inn1['grade'] = df_inn1['name'].map(lambda s: opt(s.split(', '), 1))
df_inn1['name'] = df_inn1['name'].map(lambda s: opt(s.split(', '), 0))
df_inn1.to_csv('innopolis_1.csv', index=False)
df_inn1

,name,reg,total,grade
0,Коробко Андрей,Республика Беларусь,485,11
1,Алоян Эдгар,,476,NaN
2,Коновнин Лев,г. Москва,476,11
3,Халтурин Андрей,Калининградская область,476,10
4,Карташов Владислав,г. Москва,449,10
...,...,...,...,...
797,Шиянов Пётр,г. Москва,20,8
798,Шумилов Дмитрий,Кировская область,20,9
799,Яцкевич Севастьян,Самарская область,20,10
800,Якушев Роман,,18,NaN


In [285]:
split_name(trim_cols(11, idk4(inn1[1])[1:]))

['Алоян Эдгар, ', '100', '100', '100', '76', '100', '476']

In [89]:
with open('innopolis_2') as f:
    inn2 = f.readlines()
df_inn2 = pd.DataFrame(map(lambda s: compr(idk(split_name(trim_cols(11, idk4(s)[1:])))), inn2), columns=['name', 'reg', 'total'])
df_inn2['grade'] = df_inn2['name'].map(lambda s: opt(s.split(', '), 1))
df_inn2['name'] = df_inn2['name'].map(lambda s: opt(s.split(', '), 0))
df_inn2.to_csv('innopolis_2.csv', index=False)
df_inn2

,name,reg,total,grade
0,Машира Даниил,Республика Казахстан,446,10
1,Правосудов Даниил,Самарская область,419,9
2,Альмиев Карим,Республика Татарстан (Татарстан),400,11
3,Миннахметов Искандер,Республика Татарстан (Татарстан),393,11
4,Звездин Владимир,Челябинская область,380,9
...,...,...,...,...
809,Мустафин Камиль,Республика Татарстан (Татарстан),12,9
810,Редькин Владислав,Ставропольский край,12,9
811,Карпухин Иван,г. Москва,7,9
812,Кругляк Мария,Московская область,7,11


In [90]:
with open('innopolis_3') as f:
    inn3 = f.readlines()
df_inn3 = pd.DataFrame(map(lambda s: compr(idk(split_name(trim_cols(11, idk4(s)[1:])))), inn3), columns=['name', 'reg', 'total'])
df_inn3['grade'] = df_inn3['name'].map(lambda s: opt(s.split(', '), 1))
df_inn3['name'] = df_inn3['name'].map(lambda s: opt(s.split(', '), 0))
df_inn3.to_csv('innopolis_3.csv', index=False)
df_inn3

,name,reg,total,grade
0,Звездин Владимир Сергеевич,Санкт-Петербург,519,11
1,Кардаш Кирилл Олегович,Республика Беларусь,519,10
2,Кондрашин Всеволод Юрьевич,Свердловская область,519,10
3,Насыбуллин Ильяс Ринатович,Республика Татарстан (Татарстан),519,10
4,Устименко Артём Витальевич,г. Москва,519,11
...,...,...,...,...
255,Трескова Снежана Николаевна,Ленинградская область,0,11
256,Халтурин Андрей Александрович,Калининградская область,0,11
257,Чубченко Артемий Иванович,Новосибирская область,0,10
258,Шель Ева Константиновна,г. Москва,0,10


In [100]:
with open('itmo_8') as f:
    itmo = f.readlines()
df_itmo = pd.DataFrame(map(lambda s: compr(split_name_n(idk4(s)[1:-1])), itmo), columns=['name', 'reg', 'result'])
df_itmo['reg'] = df_itmo['reg'].map(get_region)
df_itmo['result'] = df_itmo['result'].map(parse_result)
df_itmo['grade'] = 8
df_itmo8 = df_itmo
with open('itmo_9') as f:
    itmo = f.readlines()
df_itmo = pd.DataFrame(map(lambda s: compr(split_name_n(idk4(s)[1:-1])), itmo), columns=['name', 'reg', 'result'])
df_itmo['reg'] = df_itmo['reg'].map(get_region)
df_itmo['result'] = df_itmo['result'].map(parse_result)
df_itmo['grade'] = 9
df_itmo9 = df_itmo
with open('itmo_10') as f:
    itmo = f.readlines()
df_itmo = pd.DataFrame(map(lambda s: compr(split_name_n(idk4(s)[1:-1])), itmo), columns=['name', 'reg', 'result'])
df_itmo['reg'] = df_itmo['reg'].map(get_region)
df_itmo['result'] = df_itmo['result'].map(parse_result)
df_itmo['grade'] = 10
df_itmo10 = df_itmo
with open('itmo_11') as f:
    itmo = f.readlines()
df_itmo = pd.DataFrame(map(lambda s: compr(split_name_n(idk4(s)[1:-1])), itmo), columns=['name', 'reg', 'result'])
df_itmo['reg'] = df_itmo['reg'].map(get_region)
df_itmo['result'] = df_itmo['result'].map(parse_result)
df_itmo['grade'] = 11
df_itmo11 = df_itmo
df_itmo = pd.concat([df_itmo8, df_itmo9, df_itmo10, df_itmo11])
df_itmo.to_csv('itmo.csv', index=False)
df_itmo

,name,reg,result,grade
0,Сисьмекова Злата Артемовна,Свердловская область,3,8
1,Щукин Никита Александрович,Москва,3,8
2,Зеленкин Кирилл Викторович,Москва,3,8
3,Бисярин Андрей Евгеньевич,Санкт-Петербург,2,8
4,Русанов Николай Алексеевич,Новосибирская область,2,8
...,...,...,...,...
48,Селин Андрей Сергеевич,Санкт-Петербург,1,11
49,Терентьев Александр Павлович,Саратовская область,1,11
50,Удалов Макар Сергеевич,Москва,1,11
51,Усов Дмитрий Андреевич,Ставропольский край,1,11


In [92]:
with open('vuz_akadem') as f:
    vuz = f.readlines()
df_vuz = pd.DataFrame([i[:3] for i in map(lambda s: idk6(idk12(remove_times(split_name_n(idk4(s)[1:])))), vuz) if i is not None], columns=['name', 'reg', 'total'])
df_vuz['name'] = df_vuz['name'].map(lambda s: s[:-1])
df_vuz['grade'] = df_vuz['reg'].map(lambda s: opt(s.split(', '), 1)).map(lambda s: str(s).split()[0] if s else np.nan)
df_vuz['reg'] = df_vuz['reg'].map(lambda s: s.split(', ')[0])
df_vuz.to_csv('vuz_akadem.csv', index=False)
df_vuz

,name,reg,total,grade
0,Мухамадиев Дабир Юрисович,Республика Татарстан,604,10
1,Котлечков Егор Владимирович,Свердловская область,594,11
2,Андреев Артём Андреевич,Ростовская область,585,11
3,Лободов Матвей Олегович,Пермский край,570,10
4,Кондрашин Всеволод Юрьевич,Свердловская область,548,10
...,...,...,...,...
161,Чернышов Егор Сергеевич,Липецкая область,38,11
162,Караев Ян Ярославич,Воронежская область,9,11
163,Жук Владимир Олегович,Калининградская область,0,11
164,Юрасов Алексей Владимирович,Московская область,0,10


In [98]:
with open('vuz_akadem_24') as f:
    vuz_24 = f.readlines()
df_vuz24 = pd.DataFrame(map(lambda s: s.split('\t')[1:], vuz_24), columns=['name', 'reg', 'total', 'result'])
df_vuz24['result'] = df_vuz24['result'].map(parse_result)
df_vuz24.to_csv('vuz_akadem_24.csv', index=False)
df_vuz24

,name,reg,total,result
0,Габитов Шамиль Ильдарович,Республика Башкортостан,735,3
1,Ивченко Андрей Дмитриевич,Свердловская область,733,3
2,Софронов Алексей Евгеньевич,Тюменская область,717,3
3,Котлечков Егор Владимирович,Свердловская область,680,3
4,Венидиктов Тимофей Михайлович,Челябинская область,660,3
5,Андрианов Максим Игоревич,Вологодская область,655,3
6,Щемеров Игорь Сергеевич,Республика Мордовия,640,3
7,Глазман Константин Денисович,Удмуртская Республика,620,3
8,Гришечкин Иван Дмитриевич,Челябинская область,610,3
9,Лашкин Андрей Сергеевич,Нижегородская область,610,3


In [101]:
with open('itmo_8_24') as f:
    itmo = f.readlines()
df_itmo = pd.DataFrame(map(lambda s: compr(split_name_n(idk4(s)[1:-1])), itmo), columns=['name', 'reg', 'result'])
df_itmo['reg'] = df_itmo['reg'].map(get_region)
df_itmo['result'] = df_itmo['result'].map(parse_result)
df_itmo['grade'] = 8
df_itmo8 = df_itmo
with open('itmo_9_24') as f:
    itmo = f.readlines()
df_itmo = pd.DataFrame(map(lambda s: compr(split_name_n(idk4(s)[1:-1])), itmo), columns=['name', 'reg', 'result'])
df_itmo['reg'] = df_itmo['reg'].map(get_region)
df_itmo['result'] = df_itmo['result'].map(parse_result)
df_itmo['grade'] = 9
df_itmo9 = df_itmo
with open('itmo_10_24') as f:
    itmo = f.readlines()
df_itmo = pd.DataFrame(map(lambda s: compr(split_name_n(idk4(s)[1:-1])), itmo), columns=['name', 'reg', 'result'])
df_itmo['reg'] = df_itmo['reg'].map(get_region)
df_itmo['result'] = df_itmo['result'].map(parse_result)
df_itmo['grade'] = 10
df_itmo10 = df_itmo
with open('itmo_11_24') as f:
    itmo = f.readlines()
df_itmo = pd.DataFrame(map(lambda s: compr(split_name_n(idk4(s)[1:-1])), itmo), columns=['name', 'reg', 'result'])
df_itmo['reg'] = df_itmo['reg'].map(get_region)
df_itmo['result'] = df_itmo['result'].map(parse_result)
df_itmo['grade'] = 11
df_itmo11 = df_itmo
df_itmo = pd.concat([df_itmo8, df_itmo9, df_itmo10, df_itmo11])
df_itmo.to_csv('itmo_24.csv', index=False)
df_itmo

,name,reg,result,grade
0,Киселев Андрей Евгеньевич,Орловская область,3,8
1,Богданов Никита Антонович,Свердловская область,3,8
2,Снохинов Георгий Владимирович,Саратовская область,3,8
3,Ганичев Филипп Андреевич,Кировская область,2,8
4,Козлов Демид Сергеевич,Санкт-Петербург,2,8
...,...,...,...,...
82,Тундыков Сергей Сергеевич,Республика Мордовия,1,11
83,Черноморцев Глеб Вячеславович,Москва,1,11
84,Шиянов Александр Олегович,Москва,1,11
85,Щипунова Дарья Дмитриевна,Челябинская область,1,11


In [117]:
inn1[0].split('\t')

['1 ',
 'Звездин Владимир, 9 ',
 '100 ',
 '100 ',
 '100 ',
 '15 ',
 '100 ',
 '415 ',
 'Invited\n']

In [124]:
with open('innopolis_1_24') as f:
    inn1 = f.readlines()
df_inn1 = pd.DataFrame(map(lambda s: idk(s.split(' \t')[1:-1]), inn1), columns=['name', 'total'])
df_inn1['grade'] = df_inn1['name'].map(lambda s: opt(s.split(', '), 1))
df_inn1['name'] = df_inn1['name'].map(lambda s: opt(s.split(', '), 0))
df_inn1.to_csv('innopolis_1_24.csv', index=False)
df_inn1

,name,total,grade
0,Звездин Владимир,415,9
1,Сокольников Алексей,400,11
2,Тимофеев Всеволод,375,10
3,Дьяконов Сергей,372,10
4,Валиуллин Данис,370,11
...,...,...,...
447,Ишимцев Егор,9,10
448,Кондратов Егор,9,11
449,Ладоха Матвей,9,10
450,Леонов Игорь,9,8


In [125]:
with open('innopolis_2_24') as f:
    inn2 = f.readlines()
df_inn2 = pd.DataFrame(map(lambda s: idk(s.split(' \t')[1:-1]), inn2), columns=['name', 'total'])
df_inn2['grade'] = df_inn2['name'].map(lambda s: opt(s.split(', '), 1))
df_inn2['name'] = df_inn2['name'].map(lambda s: opt(s.split(', '), 0))
df_inn2.to_csv('innopolis_2_24.csv', index=False)
df_inn2

,name,total,grade
0,Gatev Alexander,500,NaN
1,Кардаш Кирилл,500,9
2,Зедгенизова Маргарита,481,11
3,Фортунатов Василий,432,11
4,Алексеев Дмитрий,427,11
...,...,...,...
1180,Гладких Дарья,10,11
1181,Живайкина Ксения,10,8
1182,Запискин Михаил,10,NaN
1183,Самрин Илья,10,NaN


In [136]:
with open('innopolis_3_24') as f:
    inn3 = f.readlines()
df_inn3 = pd.DataFrame(map(lambda s: idk(s.split('\t')[1:-1]), inn3), columns=['name', 'total'])
df_inn3['grade'] = df_inn3['name'].map(lambda s: opt(s.split(', '), 1))
df_inn3['reg'] = df_inn3['name'].map(lambda s: opt(s.split(', '), 2))
df_inn3['name'] = df_inn3['name'].map(lambda s: opt(s.split(', '), 0))
df_inn3.to_csv('innopolis_3_24.csv', index=False)
df_inn3

,name,total,grade,reg
0,Звездин Владимир,500,10,Санкт-Петербург
1,Кардаш Кирилл,500,9,Гомель
2,Савицкий Егор,465,11,Светлогорск
3,Свирид Егор,448,11,Москва
4,Ходыкин Тимофей,432,11,Санкт-Петербург
...,...,...,...,...
201,Шиляев Вадим,56,11,Орёл
202,Нурисламов Амир,40,11,Казань
203,Петров Дмитрий,28,11,Москва
204,Семейкин Андрей,24,11,Москва


In [144]:
with open('belchonok_24') as f:
    bel = f.readlines()
df_bel = pd.DataFrame(map(lambda s: ensure_cols(3, idk(split_name(idk4(s)))), bel[4:]), columns=['name', 'total', 'result'])
df_bel = df_bel[(df_bel['name'] != '') & (df_bel['name'] != 'Работа') & (df_bel['name'] != 'ID') & (df_bel['name'] != 'Статус')]
df_bel['result'] = df_bel['result'].map(parse_result)
df_bel.to_csv('belchonok_24.csv', index=False)
df_bel

,name,total,result
0,Сибагатуллина Нелли Ильшатовна,94,3
1,Акимов Аким Алексеевич,66,3
2,Данилов Анатолий Дмитриевич,62,3
3,Петров Ярослав Иванович,62,3
4,Глебов Марк Андреевич,60,3
...,...,...,...
1625,Ахметшин Эмиль Русланович,61,1
1626,Лысаков Владимир Константинович,60,1
1627,Тихомирова Дарья Евгеньевна,60,1
1628,Волычев Кирилл Максимович,58,1


In [161]:
with open('open') as f:
    opn = f.readlines()
d = {'Первой степени\n': 3, 'Второй степени\n': 2, 'Третей степени\n': 1, 'Silver meadal\n': 2, 'Bronse medal\n': 1, '\n': 0, None: 0}
df_opn = pd.DataFrame(map(lambda s: s.split('\t')[1:], opn), columns=['name', 'grade', 'participation_grade', 'reg', 'A1', 'B1', 'C1', 'D1', 'A2', 'B2', 'C2', 'D2', 'total', 'result'])
df_opn['result'] = df_opn['result'].map(d.__getitem__)
df_opn.to_csv('open.csv', index=False)
df_opn

,name,grade,participation_grade,reg,A1,B1,C1,D1,A2,B2,C2,D2,total,result
0,Кузнецов Иван Александрович,11,11,Санкт-Петербург,30,100,100,100,95,100,100,100,725,3
1,Звездин Владимир Сергеевич,11,11,Санкт-петербург,30,100,100,100,84,100,100,100,714,3
2,Муханов Сергей Дмиртиевич,11,11,Москва,30,100,82,100,91,100,100,100,703,3
3,Кардаш Кирилл Олегович,10,11,Гомель,30,100,100,100,68,100,100,100,698,3
4,Жиганов Владислав Александрович,11,11,Москва,30,100,100,100,98,100,100,50,678,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
466,Лесных Андрей Максимович,10,11,Москва,,0,,,,17,13,0,30,0
467,Андрюшкин Алдар,11,11,Москва,,15,,,,,,,15,0
468,Постникова София Вадимовна,9,11,Москва,0,0,,,,8,,0,8,0
469,Кормачев Дмитрий Иванович,11,11,Москва,7,0,,,,,,,7,0


In [160]:
with open('open_24') as f:
    opn = f.readlines()
d = {'Первой степени\n': 3, 'Второй степени\n': 2, 'Третей степени\n': 1, 'Silver meadal\n': 2, 'Bronse medal\n': 1, '\n': 0, None: 0, '': 0}
df_opn = pd.DataFrame(map(lambda s: s.split('\t')[1:], opn), columns=['name', 'grade', 'participation_grade', 'reg', 'A1', 'B1', 'C1', 'D1', 'A2', 'B2', 'C2', 'D2', 'total', 'result'])
df_opn['result'] = df_opn['result'].map(d.__getitem__)
df_opn.to_csv('open_24.csv', index=False)
df_opn

,name,grade,participation_grade,reg,A1,B1,C1,D1,A2,B2,C2,D2,total,result
0,Лосев Пётр Валерьевич,11,11,Москва,83,100,100,100,78,100,100,100,761,3
1,Абдуллин Гимран Маратович,11,11,Казань,100,100,100,100,45,100,100,46,691,3
2,Звездин Владимир Сергеевич,10,11,Санкт-Петербург,83,100,100,100,45,100,100,52,680,3
3,Грекова Дарья Сергеевна,11,11,Москва,93,72,100,100,45,100,100,22,632,3
4,Цыбань Лев Евгеньевич,10,11,Москва,36,62,100,100,32,100,100,100,630,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
424,Копьев Данил Андреевич,11,11,Петропавловск-Камчатский,,12,0,14,,,,,26,0
425,Булынко Георгий Александрович,10,11,Минск,11,0,,,,,,,11,0
426,Хотиловский Андрей Сергеевич,11,11,Москва,,4,,,,,,,4,0
427,Иванов Пётр Игоревич,10,11,Москва,,4,,0,,,,,4,0
